In [ ]:
!pip install -q torch torchvision faiss-cpu transformers==4.57.0 pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.7 MB/s eta 0:00:00


In [32]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

clip_device = "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(clip_device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [33]:
import json

with open("/content/data.json", "r") as f:
    data = json.load(f)

In [34]:
image_corpus = []
text_corpus = []

In [35]:
import numpy as np
from PIL import Image

image_embeddings = []
text_embeddings = []

for item in data:
    # ---- Image embedding ----
    try:
        image = Image.open("/content/images/" + item["image"]).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt").to(clip_device)

        with torch.no_grad():
            # Explicitly get the vision model output and extract pooler_output tensor
            outputs = clip_model.get_image_features(**inputs)
            # If get_image_features returns the tensor directly, use it; otherwise check for pooler_output
            img_emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

        image_embeddings.append(img_emb.cpu().numpy().flatten())

        # ---- Text embedding ----
        text = item["question"] + " " + item["context"]
        inputs = clip_processor(text=[text], return_tensors="pt", padding=True).to(clip_device)

        with torch.no_grad():
            # Explicitly get the text model output and extract pooler_output tensor
            outputs = clip_model.get_text_features(**inputs)
            txt_emb = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs

        text_embeddings.append(txt_emb.cpu().numpy().flatten())
    except Exception as e:
        print(f"Error processing item {item.get('id')}: {e}")

image_embeddings = np.array(image_embeddings).astype("float32")
text_embeddings = np.array(text_embeddings).astype("float32")

In [36]:
import faiss

faiss.normalize_L2(image_embeddings)
faiss.normalize_L2(text_embeddings)

In [37]:
dim = image_embeddings.shape[1]

image_index = faiss.IndexFlatIP(dim)
text_index = faiss.IndexFlatIP(dim)

image_index.add(image_embeddings)
text_index.add(text_embeddings)

In [ ]:
def retrieve_independent(query, top_k=3):

    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(clip_device)

    with torch.no_grad():
        q_emb = clip_model.get_text_features(**inputs)

    q_emb = q_emb.cpu().numpy().astype("float32")
    faiss.normalize_L2(q_emb)

    # Search separately
    D_img, I_img = image_index.search(q_emb, top_k)
    D_txt, I_txt = text_index.search(q_emb, top_k)

    images = [image_corpus[i] for i in I_img[0]]
    contexts = [text_corpus[i] for i in I_txt[0]]

    return images, contexts

In [47]:
def retrieve_with_scores(query, top_k=3):

    inputs = clip_processor(text=[query], return_tensors="pt", padding=True).to(clip_device)

    with torch.no_grad():
        q_emb = clip_model.get_text_features(**inputs)

    q_emb = q_emb.cpu().numpy().astype("float32")
    faiss.normalize_L2(q_emb)

    D_img, I_img = image_index.search(q_emb, top_k)
    D_txt, I_txt = text_index.search(q_emb, top_k)

    return D_img, I_img, D_txt, I_txt

In [ ]:
query = "in which day, the number of push-ups is lowest?"
D_img, I_img, D_txt, I_txt = retrieve_with_scores(query)
print(D_img)
print(I_img)

In [ ]:
print(D_txt)
print(I_txt)

In [42]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct", dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

In [43]:
def image_response(query, image):
  messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "image",
                  "image": image,
              },
              {"type": "text", "text": query},
          ],
      }
  ]

  # Preparation for inference
  inputs = processor.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_dict=True,
      return_tensors="pt"
  )
  inputs = inputs.to("cuda")

  # Inference: Generation of the output
  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_ids_trimmed = [
      out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]
  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  return output_text

In [44]:
def select_best_modality(D_img, I_img, D_txt, I_txt):

    # Best scores
    best_img_score = D_img[0][0]
    best_txt_score = D_txt[0][0]

    if best_img_score > best_txt_score:
        return {
            "type": "image",
            "data": data[I_img[0][0]]["image"],
            "score": best_img_score
        }
    else:
        return {
            "type": "text",
            "data": data[I_txt[0][0]]["context"],
            "score": best_txt_score
        }

In [45]:
def generate_answer(query, selected):

    if selected["type"] == "image":
        image = Image.open("/content/images"+selected["data"]).convert("RGB")

        prompt = query
        return prompt

    else:
        context = selected["data"]["context"]

        prompt = f"""
        Context: {context}
        {query}
        """

        return prompt

In [48]:
import pandas as pd
from PIL import Image

results = []

for item in data:
    q = item["question"]
    gt = item["answer"]
    idx = item["id"]

    # ---- Retrieve ----
    D_img, I_img, D_txt, I_txt = retrieve_with_scores(q)
    selected = select_best_modality(D_img, I_img, D_txt, I_txt)

    # ---- Generate Answer ----
    if selected["type"] == "image":
        img_path = "/content/images/" + selected["data"]
        img = Image.open(img_path).convert("RGB")
        mmrag_ans = image_response(q, img)[0]   # your Qwen function
    else:
        context = selected["data"]
        mmrag_ans = context   # base version (text-only response)

    results.append({
        "id": idx,
        "question": q,
        "actual_ans": gt,
        "mmrag_ans": mmrag_ans
    })

# ---- Save CSV ----
df = pd.DataFrame(results)
df.to_csv("mmrag_results.csv", index=False)

print("Saved: mmrag_results.csv")
df.head()

Saved: mmrag_results.csv


,id,question,actual_ans,mmrag_ans
0,1,At how many units of output does the company b...,500 units,"The company is making a loss of $12,500."
1,2,What is the profit or loss at 300 units of out...,"-$2,000",The profit at this point is negative.
2,3,What is the sales revenue at 600 units?,"$12,000",The profit at this point is negative.
3,4,"In the high-risk condition, which reward type ...",Large reward,Medium rewards dominate across all categories ...
4,5,Which age group has a higher framing score for...,Young Adult,Adolescents consistently outperform young adul...
